# NOTEBOOK 01: DATA MERGING & FEATURE ENGINEERING (OULAD & KAGGLE S2)

**Dự án:** Hệ thống Cảnh báo sớm Học vụ và Mô phỏng Kịch bản Học tập 2 Tầng (What-If Simulator)

**Mục tiêu Notebook 1:**
- Gộp 4 bảng cốt lõi của bộ dữ liệu OULAD (`studentInfo`, `assessments`, `studentAssessment`, `studentVle`).
- Áp dụng quy chuẩn đào tạo tín chỉ Việt Nam (tỷ lệ điểm Quá trình - Cuối kỳ 40-60 / 50-50, thang điểm 10).
- Xây dựng 3 đặc trưng học thuật chuyên sâu: `weighted_final_score`, `gpa_credits`, `gap_process_exam`.
- Xử lý bộ dữ liệu Tầng 2 (`StudentPerformanceFactors.csv`) và đồng bộ thang điểm 10.
- **Nguyên tắc kỹ thuật sống còn:** Tối ưu bộ nhớ RAM (Groupby trước -> Merge sau), chống rò rỉ dữ liệu (Anti-Data Leakage bằng cách loại bỏ bài thi Exam khỏi điểm quá trình).

---
## 📦 EPIC 1: DATA ENGINEERING & FEATURE ENGINEERING
### 🔹 Task 1.1: Kiểm tra toàn vẹn khóa chính & Làm sạch bảng OULAD gốc

**Mục tiêu:**
1. Xác thực bộ 3 khóa định danh composite `KEYS = ['id_student', 'code_module', 'code_presentation']` trên `studentInfo.csv` không bị trùng lặp.
2. Xử lý các dòng điểm bị thiếu (`NaN`) trong `studentAssessment.csv` bằng cách điền `0.0` (phản ánh sinh viên bỏ không nộp bài).
3. **Chống Data Leakage:** Loại trừ toàn bộ các bài đánh giá có `assessment_type == 'Exam'` khỏi điểm quá trình, vì sinh viên bỏ học (`Withdrawn`) sẽ không có điểm thi cuối kỳ. Nếu giữ lại điểm Exam, mô hình sẽ bị rò rỉ nhãn.
4. Ghép (`merge`) bảng điểm với `assessments.csv` để lấy thông tin môn học, hạn nộp (`date`) và trọng số (`weight`).
5. Quy đổi toàn bộ điểm về **thang điểm 10** (`score / 10.0`).
6. Chia quá trình học tập thành 2 giai đoạn: **Nửa đầu kỳ (`early`: date <= 100)** và **Nửa cuối kỳ (`late`: date > 100)** để chuẩn bị cho việc tính độ lệch phong độ.

In [ ]:
# CELL 1: Import thư viện và thiết lập cấu hình
import os
import gc
import pandas as pd
import numpy as np

# Bộ 3 khóa liên kết chuẩn trên OULAD
KEYS = ['id_student', 'code_module', 'code_presentation']
RAW_S1 = os.path.join('data', 'raw', 'S1')
RAW_S2 = os.path.join('data', 'raw', 'S2')

print("✅ Import thư viện thành công!")
print(f"   Pandas version: {pd.__version__}")
print(f"   Numpy version:  {np.__version__}")

### 1.1.1. Kiểm tra toàn vẹn khóa chính trên `studentInfo.csv`

Trong OULAD, một sinh viên (`id_student`) có thể đăng ký nhiều môn (`code_module`) và qua nhiều học kỳ (`code_presentation`). Do đó, bản ghi chỉ duy nhất khi xét trên cả bộ 3 khóa này.

In [ ]:
# CELL 2: Nạp studentInfo và kiểm tra trùng lặp khóa chính
df_info = pd.read_csv(os.path.join(RAW_S1, 'studentInfo.csv'))

# Kiểm tra số lượng bản ghi trùng khóa
dup_count = df_info.duplicated(subset=KEYS).sum()
assert dup_count == 0, f"❌ Lỗi: Có {dup_count} dòng trùng lặp khóa chính!"

print(f"✅ Bảng studentInfo.csv: {len(df_info):,} dòng.")
print(f"✅ Toàn vẹn khóa chính KEYS={KEYS}: {dup_count} trùng lặp (100% Unique).")
print(f"   Số lượng sinh viên duy nhất: {df_info['id_student'].nunique():,}")
print(f"   Số lượng môn học: {df_info['code_module'].nunique()} môn ({list(df_info['code_module'].unique())})")

### 1.1.2. Làm sạch bảng điểm & Chống Rò rỉ Dữ liệu (Anti-Data Leakage)

> ⚠️ **Cảnh báo Data Leakage:** Bảng `assessments.csv` chứa 3 loại bài kiểm tra:
> - `TMA` (Tutor Marked Assessment): Bài tập quá trình do giảng viên chấm.
> - `CMA` (Computer Marked Assessment): Bài kiểm tra trắc nghiệm máy tính chấm.
> - `Exam`: Bài thi kết thúc học phần.
>
> Trong thực tế OULAD, sinh viên bỏ học (`Withdrawn`) hầu hết rút lui trước hoặc trong kỳ thi, nên không có điểm thi `Exam`. Nếu đưa bài thi `Exam` vào tính điểm quá trình để dự báo bỏ học, mô hình sẽ bị rò rỉ dữ liệu (chỉ cần nhìn thấy thiếu điểm Exam là đoán ngay bỏ học). Vì vậy, ta **lọc bỏ hoàn toàn các bài Exam** và chỉ giữ lại bài quá trình (`TMA` và `CMA`).

In [ ]:
# CELL 3: Đọc bảng điểm, loại bỏ Exam và quy đổi về thang điểm 10
df_assess = pd.read_csv(os.path.join(RAW_S1, 'assessments.csv'))
df_sa = pd.read_csv(os.path.join(RAW_S1, 'studentAssessment.csv'))

print(f"Tổng số bài kiểm tra cấu hình trong assessments.csv: {len(df_assess):,}")
print(f"Phân bố loại bài:\n{df_assess['assessment_type'].value_counts()}\n")

# Xử lý missing values trong điểm số: sinh viên có tên trong danh sách nhưng không nộp bài -> điền 0.0
nan_scores = df_sa['score'].isna().sum()
df_sa['score'] = df_sa['score'].fillna(0.0)
print(f"✅ Đã xử lý {nan_scores} dòng điểm khuyết thiếu (NaN) -> điền 0.0")

# [ANTI-DATA LEAKAGE] Loại bỏ toàn bộ bài kiểm tra loại 'Exam'
df_assess_process = df_assess[df_assess['assessment_type'].isin(['TMA', 'CMA'])].copy()
num_exam_removed = (df_assess['assessment_type'] == 'Exam').sum()
print(f"🛡️ [ANTI-DATA LEAKAGE] Đã loại bỏ {num_exam_removed} bài Exam khỏi dữ liệu quá trình.")
print(f"   Chỉ giữ lại {len(df_assess_process)} bài đánh giá quá trình (TMA & CMA).")

# Merge lấy thông tin môn học, kỳ học, hạn nộp (date) và trọng số (weight)
df_sa_full = pd.merge(
    df_sa,
    df_assess_process[['id_assessment', 'code_module', 'code_presentation', 'assessment_type', 'date', 'weight']],
    on='id_assessment',
    how='inner'  # Inner join tự động loại trừ các bài thi Exam
)

# Quy đổi điểm từ thang 100 về thang điểm 10 chuẩn Việt Nam
df_sa_full['score_10'] = (df_sa_full['score'] / 10.0).round(2)

# Chia 2 giai đoạn học kỳ: Early (date <= 100) vs Late (date > 100)
# Ngày 100 tương ứng khoảng kết thúc nửa đầu học kỳ (học kỳ OULAD kéo dài ~240-269 ngày)
df_sa_full['phase'] = np.where(df_sa_full['date'] <= 100, 'early', 'late')

# ============================================================
# CHECKPOINTS KIỂM ĐỊNH CHẤT LƯỢNG (ACCEPTANCE CRITERIA)
# ============================================================
assert (df_sa_full['assessment_type'] == 'Exam').sum() == 0, "❌ Lỗi: Còn sót bài thi Exam!"
assert df_sa_full['score_10'].between(0.0, 10.0).all(), "❌ Lỗi: Điểm số vượt ngoài khoảng [0, 10]!"

print("\n" + "="*60)
print("🎉 TASK 1.1 CHECKPOINTS: TẤT CẢ TIÊU CHÍ ĐỀU ĐẠT CHUẨN!")
print(f"   Tổng số lượt nộp bài quá trình: {len(df_sa_full):,} dòng")
print(f"   Thang điểm score_10: min={df_sa_full['score_10'].min()}, max={df_sa_full['score_10'].max()}, mean={df_sa_full['score_10'].mean():.2f}")
print(f"   Phân bố giai đoạn nộp bài:\n{df_sa_full['phase'].value_counts()}")
print("="*60)